# 04 — Regime Analysis
**PCA Macro Slowdown Indicator**
Siddhanth Yadav · Kavin Dhanasekar · Sudhan Adithya

This notebook covers:
- Smoothed PC1 and threshold-based slowdown classification
- Dual-panel regime chart vs. NBER recessions
- Overlap statistics (recall/precision vs. NBER)
- [Optional] Logistic recession probability model

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv('../.env')

import config
from pca.regime import smooth_pc1, classify_regime, regime_periods, nber_recession_periods, compare_with_nber
from viz.charts import plot_regime_periods

pc1       = pd.read_csv('../outputs/pc1_series.csv', index_col=0, parse_dates=True).squeeze()
panel_std = pd.read_csv('../data/processed/panel_clean.csv', index_col=0, parse_dates=True)
nber      = panel_std.get('nber_recession')

print('PC1 loaded:', len(pc1), 'observations')

In [ ]:
# --- Smooth and classify ---
pc1_smooth = smooth_pc1(pc1, window=config.SMOOTHING_WINDOW)
regime     = classify_regime(pc1, threshold=config.REGIME_THRESHOLD)
slow_p     = regime_periods(regime)
rec_p      = nber_recession_periods(nber) if nber is not None else []

print(f'Slowdown periods identified: {len(slow_p)}')
for s, e in slow_p[:5]:
    print(f'  {s.date()} → {e.date()}')

In [ ]:
# --- Dual-panel regime chart ---
fig, axes = plot_regime_periods(pc1, pc1_smooth, slow_p, rec_p, threshold=config.REGIME_THRESHOLD)
plt.tight_layout()
plt.show()

In [ ]:
# --- NBER comparison stats ---
if nber is not None:
    nber_aligned = nber.reindex(regime.index).fillna(0)
    comp = compare_with_nber(regime, nber_aligned)
    print('\nNBER vs. PCA Slowdown Regime:')
    print(comp)

In [ ]:
# --- Sensitivity: vary threshold ---
thresholds = [-1.0, -0.5, 0.0, 0.5]
fig, axes  = plt.subplots(len(thresholds), 1, figsize=(14, 3 * len(thresholds)), sharex=True)

for ax, thr in zip(axes, thresholds):
    r = classify_regime(pc1, threshold=thr)
    ax.fill_between(r.index, 0, r.values, color=config.SLOWDOWN_SHADE_COLOR, step='post', alpha=0.8)
    ax.plot(pc1.index, pc1.values, color='#2c7bb6', linewidth=0.8)
    ax.axhline(thr, color='red', linestyle='--', linewidth=0.8)
    ax.set_title(f'Threshold = {thr}', fontsize=10)
    ax.set_ylim(-4, 4)

plt.suptitle('Sensitivity to Regime Threshold', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## [Optional] Logistic Recession Probability Model
Run the cell below only if the optional extension is in scope.

In [ ]:
# --- [Optional] Logistic model ---
if nber is not None:
    from analysis.recession_model import prepare_logit_data, fit_logit, evaluate_model

    fin_cols  = [c for c in config.FINANCIAL_COLS if c in panel_std.columns]
    fin_panel = panel_std[fin_cols] if fin_cols else None

    X, y = prepare_logit_data(pc1, nber, financial_panel=fin_panel, lag_pc1=1)
    logit_res = fit_logit(X, y)
    metrics   = evaluate_model(logit_res, X, y)
    print(metrics)
else:
    print('NBER data not available — skipping logistic model.')